# Step 02 – Reference samples: field-recorded vs. image-interpreted points

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 2.2.3
- **Needs:** training-sample table in `DATA_DIR`; downloads the public OpenStreetMap roads for Morocco
- **Writes:** `RESULTS_DIR/R4_6_sources/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   DATA_DIR / osm/
#   RESULTS_DIR / R4_6_sources/

In [ ]:
"""
Field vs Google Earth reference points from the field-access rule

Collection protocol (as applied in the field): reference points within FIELD_DISTANCE_M of
the road/track network were recorded in the field (QField / Geo Tracker); points in less
accessible terrain were interpreted from very-high-resolution Google Earth Pro imagery.
This script applies that rule to the 2,800 reference points with the OpenStreetMap road
network (Geofabrik Morocco extract) and reports the counts per class.

It also reports the counts for several distances and the distribution of distances per
class (useful for the sampling-design description).

In Colab: geopandas is pre-installed. The road file (~ a few hundred MB) is downloaded once
to DATA_DIR/osm/ and reused afterwards.

Outputs (OUT_DIR):
  R4_6_points_road_distance.csv    every point: class, distance to road (m), assigned source
  R4_6_source_by_class.csv         counts per class at FIELD_DISTANCE_M (main table)
  R4_6_sensitivity_distances.csv   field share per class for several distances
  R4_6_distance_summary.csv        distance to road per class (median, quartiles)
"""


import os, json, subprocess, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------------ config
DATA_PATH = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
OSM_DIR   = DATA_DIR + 'osm/'
OSM_URL   = 'https://download.geofabrik.de/africa/morocco-latest-free.shp.zip'
PBF_URL   = 'https://download.geofabrik.de/africa/morocco-latest.osm.pbf'   # fallback
OUT_DIR   = RESULTS_DIR + 'R4_6_sources/'
CRS       = 'EPSG:32629'
LABEL_COL = 'class'
CLASS_NAMES = {0: 'Water', 1: 'Forest', 2: 'Bare ground', 3: 'Urban', 4: 'Crops/grass'}

# >>> set to what you actually did in the field <<<
FIELD_DISTANCE_M = 100                 # points closer than this to a road = field (collection rule)
SENSITIVITY_M    = [100, 250, 500, 1000, 2000]
# OSM road types used to reach field sites. Remove the 'track*' types if you only used
# paved roads; keep them if you also drove on dirt tracks.
ROAD_CLASSES = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified',
                'residential', 'service', 'living_street',
                'motorway_link', 'trunk_link', 'primary_link', 'secondary_link', 'tertiary_link',
                'track', 'track_grade1', 'track_grade2', 'track_grade3', 'track_grade4', 'track_grade5']

os.makedirs(OSM_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ reference points
df = pd.read_csv(DATA_PATH)
if {'longitude', 'latitude'} <= set(df.columns):
    lon, lat = df['longitude'].values, df['latitude'].values
else:
    xy = np.array([json.loads(g)['coordinates'] for g in df['.geo']])
    lon, lat = xy[:, 0], xy[:, 1]
pts = gpd.GeoDataFrame({'class': df[LABEL_COL].map(CLASS_NAMES).fillna(df[LABEL_COL].astype(str))},
                       geometry=gpd.points_from_xy(lon, lat), crs='EPSG:4326')
print('Reference points:', len(pts))

# ------------------------------------------------------------------ OSM roads (downloaded once)
def download(url, path):
    print(f'Downloading {os.path.basename(path)} (once) ...')
    subprocess.run(['wget', '-q', '--tries=3', '-O', path, url], check=True)
    print(f'   {os.path.getsize(path) / 1e6:.0f} MB')

def read_roads(bbox):
    """Roads from the Geofabrik shapefile package; falls back to the raw .osm.pbf."""
    zip_path = OSM_DIR + 'morocco-latest-free.shp.zip'
    for attempt in range(2):
        if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 1e6:
            download(OSM_URL, zip_path)
        try:
            with zipfile.ZipFile(zip_path) as z:
                names = z.namelist()
            shp = [n for n in names if 'roads' in n.lower() and n.lower().endswith('.shp')]
            if shp:
                print('Road layer:', shp[0])
                return gpd.read_file(f'zip://{zip_path}!{shp[0]}', bbox=bbox)[['fclass', 'geometry']]
            print('No roads layer in the shapefile package. Files inside:', names[:15])
            break
        except zipfile.BadZipFile:
            print('The downloaded file is not a valid zip (incomplete download?) - downloading again')
            os.remove(zip_path)
    pbf_path = OSM_DIR + 'morocco-latest.osm.pbf'
    if not os.path.exists(pbf_path) or os.path.getsize(pbf_path) < 1e6:
        download(PBF_URL, pbf_path)
    print('Reading roads from the .osm.pbf file (a few minutes) ...')
    r = gpd.read_file(pbf_path, layer='lines', bbox=bbox, columns=['highway'])
    return r[r['highway'].notna()].rename(columns={'highway': 'fclass'})[['fclass', 'geometry']]

minx, miny, maxx, maxy = pts.total_bounds
pad = 0.05                                               # ~5 km margin around the points
roads = read_roads((minx - pad, miny - pad, maxx + pad, maxy + pad))
print('Road types found:', roads['fclass'].value_counts().to_dict())
roads = roads[roads['fclass'].isin(ROAD_CLASSES)].to_crs(CRS)
print(f'Road segments used: {len(roads)}')

# ------------------------------------------------------------------ distance to nearest road
pts_utm = pts.to_crs(CRS)
near = gpd.sjoin_nearest(pts_utm, roads[['fclass', 'geometry']], how='left', distance_col='dist_m')
near = near[~near.index.duplicated(keep='first')]        # ties: keep one road per point
pts_utm['dist_road_m'] = near['dist_m'].round(1)
pts_utm['nearest_road_type'] = near['fclass']
pts_utm['source'] = np.where(pts_utm['dist_road_m'] <= FIELD_DISTANCE_M, 'Field', 'Google Earth Pro')
# Exception to the access rule: all urban points were interpreted in Google Earth Pro
pts_utm.loc[pts_utm['class'] == 'Urban', 'source'] = 'Google Earth Pro'

out = pd.DataFrame({'point': pts_utm.index, 'class': pts_utm['class'], 'lon': lon, 'lat': lat,
                    'dist_road_m': pts_utm['dist_road_m'],
                    'nearest_road_type': pts_utm['nearest_road_type'], 'source': pts_utm['source']})
out.to_csv(OUT_DIR + 'R4_6_points_road_distance.csv', index=False)

order = [c for c in CLASS_NAMES.values() if c in set(out['class'])]
tab = pd.crosstab(out['class'], out['source'], margins=True, margins_name='Total').reindex(order + ['Total'])
tab.to_csv(OUT_DIR + 'R4_6_source_by_class.csv')
print(f'\n=== Points per class and source (field = within {FIELD_DISTANCE_M} m of a road) ===')
print(tab.to_string())

sens = pd.DataFrame({f'<= {d} m': out.groupby('class')['dist_road_m'].apply(lambda v: 100 * (v <= d).mean())
                     for d in SENSITIVITY_M}).reindex(order)
sens.loc['All'] = [100 * (out['dist_road_m'] <= d).mean() for d in SENSITIVITY_M]
sens = sens.round(1)
sens.to_csv(OUT_DIR + 'R4_6_sensitivity_distances.csv')
print('\n=== % of points within each distance of a road ===\n', sens.to_string())

summ = out.groupby('class')['dist_road_m'].describe(percentiles=[0.25, 0.5, 0.75]).reindex(order)
summ = summ[['count', '25%', '50%', '75%', 'max']].round(0)
summ.to_csv(OUT_DIR + 'R4_6_distance_summary.csv')
print('\n=== Distance to the nearest road per class (m) ===\n', summ.to_string())
print('\nFiles saved to:', OUT_DIR)

In [ ]:
import pandas as pd, geopandas as gpd
D = RESULTS_DIR + 'R4_6_sources/'
f = pd.read_csv(D + 'R4_6_points_road_distance.csv')
g = gpd.GeoDataFrame(f, geometry=gpd.points_from_xy(f['lon'], f['lat']), crs='EPSG:4326')
g.to_file(D + 'R4_6_field_points.gpkg', layer='field_points', driver='GPKG')
print(len(g), 'points saved to R4_6_field_points.gpkg')

In [ ]:
import pandas as pd, geopandas as gpd
D = RESULTS_DIR + 'R4_6_sources/'

pts = pd.read_csv(D + 'R4_6_points_road_distance.csv')        # all 2,800 points
field = pts[pts['source'] == 'Field'].copy()                    # keep field points only
print('Field points per class:\n', field['class'].value_counts().to_string())
assert len(field) == 265, f'Expected 265 field points, found {len(field)}'

g = gpd.GeoDataFrame(field[['point', 'class', 'lon', 'lat', 'dist_road_m', 'nearest_road_type']],
                     geometry=gpd.points_from_xy(field['lon'], field['lat']), crs='EPSG:4326')
out = D + 'FIELD_ONLY_265_points.gpkg'
g.to_file(out, layer='field_points_only', driver='GPKG')
print(f'\nSaved {len(g)} field points to {out}')